In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np
import math

In [ ]:
print(tf.__version__)

# Load dataset

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_train.shape

In [ ]:
def visualize_samples(X, y, n=None):
    X = X.reshape(-1, 28, 28)
    if n is None:
        indices = range(len(X))
        n = len(X)
    else:
        indices = np.random.choice(len(X), n, replace=False)
    size = int(math.sqrt(n))
    plt.figure(figsize=(size, size))
    for i, idx in enumerate(indices):
        plt.subplot(size, size, i + 1)
        plt.imshow(X[idx], cmap="gray")
        plt.title(str(y[idx]))
        plt.axis("off")

    plt.tight_layout()
    plt.show()
    
visualize_samples(X_test, y_test, 16)

In [ ]:
X_train = X_train.astype('float32') / 255
X_test = X_test.astype('float32') / 255

# Encoder Definition

In [ ]:
class Encoder(keras.Model):
    def __init__(self, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.l1 = layers.Dense(512, activation='relu')
        self.l2 = layers.Dense(256, activation='relu')
        self.z_mean = layers.Dense(latent_dim)
        self.z_log_var = layers.Dense(latent_dim)
        self.z = layers.Lambda(self.reparameterize)
    @staticmethod
    def reparameterize(args):
        z_mean, z_log_var = args
        eps = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * eps
    def call(self, x):
        h = self.l1(x)
        h = self.l2(h)
        z_mean = self.z_mean(h)
        z_log_var = self.z_log_var(h)
        z = self.z([z_mean, z_log_var])
        return z_mean, z_log_var, z

# Decoder Definition

In [ ]:
class Decoder(keras.Model):
    def __init__(self, original_dim):
        super().__init__()
        self.l1 = layers.Dense(256, activation='relu')
        self.l2 = layers.Dense(512, activation='relu')
        self.out = layers.Dense(original_dim, activation='sigmoid')
    def call(self, z):
        h = self.l1(z)
        h = self.l2(h)
        x_recon = self.out(h)
        return x_recon

# VAE definition

In [ ]:
class VAE(keras.Model):
    def __init__(self, original_dim, latent_dim, noise):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(original_dim)
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")
        self.noise = noise
        self.flatten = layers.Flatten()
    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]
    def ELBOloss(self, data, reconstruction, z_mean, z_log_var):

        recon_loss = 1/(self.noise * 2) * tf.reduce_sum( tf.square(data - reconstruction), axis=1) # per sample
        recon_loss = tf.reduce_mean(recon_loss) # average over batch
        
        kl_loss = 0.5 * tf.reduce_sum(tf.square(z_mean) + - 1 - z_log_var + tf.exp(z_log_var), axis=1) # per sample
        kl_loss = tf.reduce_mean(kl_loss) # average over batch
        
        total_loss = recon_loss + kl_loss # sign is flip from derivation
        return total_loss, recon_loss, kl_loss
    def crossEntropy_ELBOloss(self, data, reconstruction, z_mean, z_log_var):

        recon_loss = tf.reduce_sum(keras.losses.binary_crossentropy(data, reconstruction)) # per sample
        recon_loss = tf.reduce_mean(recon_loss) # average over batch
        
        kl_loss = 0.5 * tf.reduce_sum(tf.square(z_mean) + - 1 - z_log_var + tf.exp(z_log_var), axis=1) # per sample
        kl_loss = tf.reduce_mean(kl_loss) # average over batch
        
        total_loss = recon_loss + kl_loss # sign is flip from derivation
        return total_loss, recon_loss, kl_loss
    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)      
            total_loss, recon_loss, kl_loss = self.ELBOloss(data, reconstruction, z_mean, z_log_var)
    
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def test_step(self, data):
        z_mean, z_log_var, z = self.encoder(data)
        reconstruction = self.decoder(z)      
        total_loss, recon_loss, kl_loss = self.ELBOloss(data, reconstruction, z_mean, z_log_var)
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }
    def call(self, x, training=False):
        z_mean, z_log_var, z = self.encoder(x, training=training)
        reconstruction = self.decoder(z, training=training)
        return reconstruction
    

# Training 

In [ ]:
latent_dim = 16
model = VAE(original_dim=28*28, latent_dim=latent_dim, noise=0.1)
model.compile(optimizer=keras.optimizers.Adam())
X_train = X_train.reshape(-1, 28*28)
X_test = X_test.reshape(-1, 28*28)
history = model.fit(X_train, epochs=30, batch_size=128, validation_data=X_test, verbose=1)

In [ ]:
def plot_loss(history, detail=False):
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    if detail:
        plt.plot(history.history['kl_loss'], label='kl_loss')
        plt.plot(history.history['reconstruction_loss'], label='reconstruction_loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()
plot_loss(history, True)

# Test Reconstruction

In [ ]:
x_recon = model.predict(X_test[16:32])
visualize_samples(X_test[16:32], y_test[16:32])
visualize_samples(x_recon, y_test[16:32])

# Test generation

In [ ]:
latent_dim = 16
z_samples = tf.random.normal(shape=(16, latent_dim))
x_gen = model.decoder(z_samples)
visualize_samples(x_gen.numpy(), range(16))